# 08 · CNN — FashionMNIST

Two conv+pool blocks, then 128-unit head with dropout.

Source: `CNN_Exercise/CNNs_Exercise.ipynb`

Course uses LogSoftmax + CrossEntropyLoss. CE already includes log-softmax, so the model below returns **logits**. Enable GPU: Runtime → Change runtime type → GPU.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=eval_transform)

train_subset_size, test_subset_size = 10000, 3000
train_subset, _ = random_split(dataset, [train_subset_size, len(dataset) - train_subset_size],
                               generator=torch.Generator().manual_seed(42))
test_subset, _ = random_split(test_dataset, [test_subset_size, len(test_dataset) - test_subset_size],
                              generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)
print(len(train_subset), len(test_subset))

In [ ]:
classes = dataset.classes
images, labels = next(iter(train_loader))
print(type(images), images.shape, labels.shape)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0].numpy(), cmap='gray')
    ax.set_title(f'Label: {labels[i].item()} ({classes[labels[i]]})', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

Architecture: Conv(1→32, 3×3) ReLU Pool → Conv(32→64, 3×3) ReLU Pool → FC 64×7×7 → 128 → dropout 0.5 → 10.

In [ ]:
class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        return self.classifier(x)

model = FashionMNISTCNN().to(device)
print(model)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.99))

In [ ]:
num_epochs = 10
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f'Accuracy on test data: {accuracy:.2f}%')

In [ ]:
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot']
images, labels = next(iter(test_loader))
images_d, labels_d = images.to(device), labels.to(device)
with torch.no_grad():
    predicted = model(images_d).argmax(dim=1).cpu()

plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f'Pred: {classes[predicted[i]]}\nTrue: {classes[labels[i]]}', fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.show()